# Búsqueda con checkpoints

In [ ]:
import pandas as pd
import time
import os
from metacontentlibraryapi import MetaContentLibraryAPIClient as client

# --- CONFIGURACIÓN DE SEGURIDAD Y API ---
client.set_default_version(client.LATEST_VERSION)

# --- CONFIGURACIÓN DE PRODUCCIÓN ---
FECHA_INICIO = "2025-01-01"
FECHA_FIN = "2026-02-15"
NOMBRE_ARCHIVO = "dataset_buap_FINAL_recuperado.csv"
META_PAGINAS = 50

# --- FASE 0: CARGAR PROGRESO PREVIO ---
ids_posts_procesados = set()
lista_datos = []

if os.path.exists(NOMBRE_ARCHIVO):
    try:
        df_previo = pd.read_csv(NOMBRE_ARCHIVO)
        # Extraemos IDs únicos de posts para no repetirlos
        if 'ID_Post' in df_previo.columns:
            ids_posts_procesados = set(df_previo['ID_Post'].unique().tolist())
        lista_datos = df_previo.to_dict('records')
        print(f"V Progreso cargado: {len(ids_posts_procesados)} posts ya procesados.")
        print(f"V Registros totales en memoria: {len(lista_datos)}")
    except Exception as e:
        print(f"X No se pudo cargar el archivo previo (empezando de cero): {e}")
else:
    print("Iniciando nueva recolección (sin archivo previo).")

# --- ESTRATEGIA DE BÚSQUEDA ---
TERMINOS_CLAVE = [
    "BUAP", "Facultad BUAP", "Preparatoria BUAP", "Confesiones BUAP",
    "Facultad de Medicina BUAP", "Facultad de Ingeniería BUAP",
    "Facultad de Derecho BUAP", "Facultad de Administracion BUAP",
    "Facultad de Arquitectura BUAP", "Facultad de Artes BUAP",
    "Facultad de Ciencias de la Comunicacion BUAP", 
    "Complejo Cultural Universitario BUAP"
]

ids_vistos_paginas = set()
paginas_para_analizar = []

print(f"\n === INICIANDO EXTRACCIÓN INCREMENTAL === ")
print(f"Periodo: {FECHA_INICIO} a {FECHA_FIN}\n")

# --- FASE 1: LOCALIZACIÓN DE PÁGINAS ---
print(f">>> FASE 1: Localizando páginas ... ")
try:
    for termino in TERMINOS_CLAVE:
        if len(paginas_para_analizar) >= META_PAGINAS: break

        response = client.get(
            path="facebook/pages/preview",
            params={"q": termino, "fields": "id, name, fan_count", "limit": 50}
        )
        data = response.json().get('data', [])

        for p in data:
            if len(paginas_para_analizar) >= META_PAGINAS: break
            pid = p['id']
            nombre = p.get('name', '').upper()

            # Filtro básico de relevancia para el corpus académico
            es_buap = any(x in nombre for x in ["BUAP", "LOBOS", "BENEMERITA", "FACULTAD", "PREPARATORIA"])

            if pid not in ids_vistos_paginas and es_buap:
                paginas_para_analizar.append(p)
                ids_vistos_paginas.add(pid)

    print(f"V Se han seleccionado {len(paginas_para_analizar)} páginas.")
except Exception as e:
    print(f"X Error en Fase 1: {e}")

# --- FASE 2: EXTRACCIÓN DE CONTENIDO ---
if paginas_para_analizar:
    total_acumulado_global = len(lista_datos)

    for i, pagina in enumerate(paginas_para_analizar):
        print(f"\n" + "="*60)
        print(f"[{i+1}/{len(paginas_para_analizar)}] PÁGINA: {pagina['name']}")
        print("="*60)

        comentarios_pagina = 0
        posts_omitidos = 0

        try:
            resp_posts = client.get(
                path="facebook/posts/preview",
                params={
                    "surface_ids": [pagina['id']],
                    "since": FECHA_INICIO,
                    "until": FECHA_FIN,
                    "fields": "id, text, created_time, comments_count",
                    "limit": 50
                }
            )

            while True:
                posts_data = resp_posts.json().get('data', [])
                if not posts_data: break

                for post in posts_data:
                    post_id = post['id']
                    
                    # --- LÓGICA DE SALTO ---
                    if post_id in ids_posts_procesados:
                        posts_omitidos += 1
                        continue 

                    try:
                        resp_com = client.get(
                            path=f"facebook/posts/{post_id}/comments/preview",
                            params={"fields": "id, text, creation_time", "limit": 50}
                        )

                        while True:
                            comentarios_data = resp_com.json().get('data', [])
                            if not comentarios_data: break

                            for c in comentarios_data:
                                lista_datos.append({
                                    "Pagina_Origen": pagina['name'],
                                    "ID_Pagina": pagina['id'],
                                    "ID_Post": post_id,
                                    "Fecha_Post": post.get('created_time'),
                                    "Texto_Post_Completo": post.get('text', ""),
                                    "ID_Comentario": c.get('id'),
                                    "Fecha_Comentario": c.get('creation_time'),
                                    "Texto_Comentario": c.get('text')
                                })
                                comentarios_pagina += 1
                                total_acumulado_global += 1

                            if client.has_next_page(resp_com):
                                resp_com = client.query_next_page(resp_com)
                                time.sleep(0.1) # Breve pausa para evitar Rate Limit
                            else:
                                break
                        
                        # Marcar post como procesado
                        ids_posts_procesados.add(post_id)

                    except Exception as e:
                        print(f" ! Error en post {post_id}: {e}")
                        continue

                if client.has_next_page(resp_posts):
                    resp_posts = client.query_next_page(resp_posts)
                    time.sleep(0.5)
                else:
                    break
            
            print(f"V Finalizado: {pagina['name']}. Nuevos: {comentarios_pagina} | Omitidos: {posts_omitidos}")
            
            # GUARDADO AUTOMÁTICO después de cada página (Checkpoint)
            pd.DataFrame(lista_datos).to_csv(NOMBRE_ARCHIVO, index=False, encoding='utf-8-sig')

        except Exception as e:
            print(f"X Error procesando página {pagina['name']}: {e}")
            continue

# --- RESULTADO FINAL ---
df_final = pd.DataFrame(lista_datos)
print("\n" + "#"*60)
print(f"PROCESO COMPLETADO. DATASET TOTAL: {len(df_final)} REGISTROS.")
print(f"Archivo actualizado: {NOMBRE_ARCHIVO}")
print("#"*60)